In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.ToTensor()
train_loader = DataLoader(datasets.MNIST(root='./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)

print("--- Variation 1: GAN on MNIST ---")
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(nn.Linear(100, 256), nn.ReLU(), nn.Linear(256, 784), nn.Tanh())
    def forward(self, x): return self.main(x)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(nn.Linear(784, 256), nn.LeakyReLU(0.2), nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, x): return self.main(x)

netG, netD = Generator().to(device), Discriminator().to(device)
criterion = nn.BCELoss()
optD, optG = optim.Adam(netD.parameters(), lr=0.0002), optim.Adam(netG.parameters(), lr=0.0002)

for i, (data, _) in enumerate(train_loader):
    real_data = data.view(-1, 784).to(device)
    batch_size = real_data.size(0)
    
    # Train D
    netD.zero_grad()
    label_real = torch.ones(batch_size, 1).to(device)
    output_real = netD(real_data)
    errD_real = criterion(output_real, label_real)
    
    noise = torch.randn(batch_size, 100).to(device)
    fake_data = netG(noise)
    label_fake = torch.zeros(batch_size, 1).to(device)
    output_fake = netD(fake_data.detach())
    errD_fake = criterion(output_fake, label_fake)
    
    errD = errD_real + errD_fake
    errD.backward()
    optD.step()
    
    # Train G
    netG.zero_grad()
    label_real_g = torch.ones(batch_size, 1).to(device)
    output_g = netD(fake_data)
    errG = criterion(output_g, label_real_g)
    errG.backward()
    optG.step()
    
    if i == 50:
        print(f"GAN Step {i} | Loss_D: {errD.item():.4f} | Loss_G: {errG.item():.4f}")
        break

print("\n--- Variation 2: Diffusion Model (Adding Noise) ---")
# Simple forward diffusion demonstration
def forward_diffusion(x0, t, beta):
    noise = torch.randn_like(x0)
    alpha = 1 - beta
    alpha_cumprod = torch.cumprod(alpha, dim=0)
    mean = torch.sqrt(alpha_cumprod[t]) * x0
    variance = torch.sqrt(1 - alpha_cumprod[t]) * noise
    return mean + variance

# Demo adding noise
sample_image = next(iter(train_loader))[0][0].view(-1, 784)
betas = torch.linspace(0.0001, 0.02, 100) # 100 timesteps
noisy_image = forward_diffusion(sample_image, t=50, beta=betas)
print(f"Original image mean: {sample_image.mean().item():.4f}, std: {sample_image.std().item():.4f}")
print(f"Noisy image (t=50) mean: {noisy_image.mean().item():.4f}, std: {noisy_image.std().item():.4f}")